In [82]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

# We'll use a consistent font size typical for IEEE figures
plt.rcParams.update({
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 10,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "mathtext.fontset": "stix",
    "font.family": "STIXGeneral",
})



def fig2a_pole_plane():
    poles = np.array([-0.2+1.2j, -0.8+1.2j, -0.5+4.0j])
    fig, ax = plt.subplots(figsize=(3.4, 2.6))
    # stable region shading (Re<0)
    # ax.axvspan(-1.2, 0, alpha=0.08)

    # ax.axvline(0, linewidth=0.2)
    # ax.axhline(0, linewidth=0.2, alpha=0.6)
    ax.scatter(poles.real, poles.imag, s=24, zorder=3)
    # labels near points
    labels=[r"$p_1$", r"$p_2$", r"$p_3$"]
    offsets=[(6,6), (-18,6), (6,6)]
    for (x,y),lab,(dx,dy) in zip(zip(poles.real,poles.imag), labels, offsets):
        ax.annotate(lab, (x,y), textcoords="offset points", xytext=(dx,dy), ha="left", va="bottom")
    # effect arrows (in axes coords to avoid overlap)
    ax.annotate("Slower decay", xy=(0.20, 0.08), xycoords="axes fraction",
                xytext=(0.70, 0.08), textcoords="axes fraction",
                ha="center", va="center",
                arrowprops=dict(arrowstyle="<-", lw=0.8))
    ax.annotate("Higher oscillation", xy=(0.06, 0.20), xycoords="axes fraction",
                xytext=(0.06, 0.70), textcoords="axes fraction",
                ha="center", va="center", rotation=90,
                arrowprops=dict(arrowstyle="<-", lw=0.8))
    ax.set_xlim(-1.1, 0.15)
    ax.set_ylim(-0.2, 5.2)
    ax.set_xlabel(r"$\Re(p)$")
    ax.set_ylabel(r"$\Im(p)$")
    ax.set_title("")  # use subcaption
    ax.grid(alpha=0.25)
    fig.tight_layout(pad=0.25)
    return fig

def prony_kernel_time(t, residues, poles):
    t=np.asarray(t)
    k=np.zeros_like(t, dtype=np.complex128)
    for r,p in zip(residues, poles):
        k += r*np.exp(p*t)
    return k.real

def fig2b_kernel_offsets():
    # Construct two complex-conjugate modes + one real mode
    t = np.linspace(0, 2, 800)
    poles = [-2.0+2.0j, -2.0-2.0j, -6.0+0j]
    residues = [1.0+0.0j, 1.0+0.0j, 0.6+0.0j]
    # individual real-valued modes (using conjugate pairs -> real)
    mode1 = prony_kernel_time(t, [residues[0], residues[1]], [poles[0], poles[1]])
    mode2 = prony_kernel_time(t, [residues[2]], [poles[2]])
    # make a third mode by changing imag frequency for illustration
    poles3 = [-1.2+6.0j, -1.2-6.0j]
    residues3 = [0.5+0j, 0.5+0j]
    mode3 = prony_kernel_time(t, residues3, poles3)
    ksum = mode1 + mode2 + mode3
    
    # normalize each mode for display
    def norm(z):
        m=np.max(np.abs(z))
        return z/m if m>0 else z
    mode1n, mode2n, mode3n, ksumn = map(norm, [mode1, mode2, mode3, ksum])
    
    offsets = [2.5, 1.2, 0.0, -1.6]
    fig, ax = plt.subplots(figsize=(3.4, 2.6))
    ax.plot(t, mode1n + offsets[0], linewidth=1.0)
    ax.plot(t, mode2n + offsets[1], linewidth=1.0)
    ax.plot(t, mode3n + offsets[2], linewidth=1.0)
    ax.plot(t, ksumn + offsets[3], linewidth=1.2)
    
    ax.set_yticks(offsets)
    # 不显示ytick
    ax.set_yticklabels([])
    # ax.set_yticklabels(["mode 1", "mode 2", "mode 3", r"$k_\phi(t)$"])
    ax.set_xlabel(r"Time (s)")
    ax.set_ylabel("")
    ax.grid(alpha=0.25)
    # small note without equations
    # ax.text(0.02, 0.96, "Prony-mode composition", transform=ax.transAxes, ha="left", va="top")
    ax.set_xlim(t.min(), t.max())
    fig.tight_layout(pad=0.25)
    return fig

def causal_conv_same(x, k):
    x=np.asarray(x); k=np.asarray(k)
    y = np.convolve(x, k, mode="full")[:len(x)]
    return y

def fig2c_signal_enhance():
    fs = 2000
    t = np.arange(0, 1.5, 1/fs)
    rng = np.random.default_rng(0)
    x = 0.25 * rng.standard_normal(len(t))
    # add a burst around 0.8s (decaying sinusoid)
    burst_center = 0.8
    burst = np.zeros_like(t)
    idx = (t>=burst_center) & (t<=burst_center+0.18)
    burst[idx] = np.exp(-20*(t[idx]-burst_center)) * np.sin(2*np.pi*80*(t[idx]-burst_center))
    x = x + 1.0*burst
    
    # kernel: resonant around 80 Hz, moderate decay
    tk = np.arange(0, 0.25, 1/fs)
    poles = [-25+2j*np.pi*80, -25-2j*np.pi*80]
    residues = [1.0+0j, 1.0+0j]
    k = prony_kernel_time(tk, residues, poles)
    # normalize kernel energy for stable display
    k = k / (np.sqrt(np.sum(k**2)) + 1e-12)
    
    y = causal_conv_same(x, k)
    # display normalization (z-score) to make comparison readable
    def z(z): 
        return (z - np.mean(z)) / (np.std(z) + 1e-12)
    xd, yd = z(x), z(y)
    
    fig, ax = plt.subplots(figsize=(4, 5.3))
    ax.plot(t, xd, linewidth=1.0, label="Input $x(t)$")
    ax.plot(t, yd, linewidth=1.0, label="Enhanced $y(t)$")
    # highlight burst region
    ax.axvspan(burst_center, burst_center+0.18, alpha=0.10)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Normalized Amplitude")
    ax.set_ylim(-6,6)
    ax.legend(loc="upper right", frameon=False)
    ax.grid(alpha=0.25)
    
    # inset zoom
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes
    axins = inset_axes(ax, width="40%", height="30%", loc="lower left", borderpad=1)
    z0, z1 = burst_center-0.02, burst_center+0.10
    sel = (t>=z0) & (t<=z1)
    axins.plot(t[sel], xd[sel], linewidth=0.9)
    axins.plot(t[sel], yd[sel], linewidth=0.9)
    axins.set_xlim(z0, z1)
    axins.set_ylim(min(xd[sel].min(), yd[sel].min())-0.2, max(xd[sel].max(), yd[sel].max())+0.2)
    axins.set_xticks([])
    axins.set_yticks([])
    ax.indicate_inset_zoom(axins, edgecolor="black", lw=0.6)
    fig.tight_layout(pad=0.25)
    return fig

fig_a = fig2a_pole_plane()
fig_b = fig2b_kernel_offsets()
fig_c = fig2c_signal_enhance()

paths=[]
for fig, name in [(fig_a,"fig2a_poles.svg"),(fig_b,"fig2b_kernel.svg"),(fig_c,"fig2c_signal.svg")]:
    p=name
    fig.savefig(p, bbox_inches="tight", pad_inches=0.02)
    paths.append(str(p))
    plt.close(fig)
paths

C:\Users\Administrator\AppData\Local\Temp\ipykernel_39120\2960569198.py:152: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(pad=0.25)


['fig2a_poles.svg', 'fig2b_kernel.svg', 'fig2c_signal.svg']

In [14]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Fig.2 subfigures generator (forward-designed to match the desired paper narrative):
(a) Pole locations -> decay/oscillation intuition (kernel poles)
(b) Operator kernel as a sum of exponential modes (stacked, minimal annotation)
(c) Input vs enhanced signal (display-normalized) with an inset zoom

Usage:
  python make_fig2_subfigs.py --outdir ./fig2

Outputs:
  fig2a_poles.pdf
  fig2b_kernel.pdf
  fig2c_signal.pdf
"""

from __future__ import annotations
import argparse
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

# ---------- global style (keep minimal & paper-friendly) ----------
mpl.rcParams.update({
    "pdf.fonttype": 42,          # embed TrueType
    "ps.fonttype": 42,
    "font.size": 9,
    "axes.titlesize": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def _ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def fig2a_poles(outpath: str) -> None:
    """
    (a) Pole locations for the operator kernel (NOT system identification):
        - stable half-plane shading (Re < 0)
        - Re=0 boundary line
        - three representative poles with short labels
        - two short directional annotations (no long paragraphs inside axis)
    """
    # Representative poles (kernel poles): isolate decay vs oscillation effects
    # p1 and p2: same Im, different Re (decay contrast)
    # p2 and p3: similar Re, larger Im (oscillation contrast)
    p1 = -0.35 + 2.0j
    p2 = -1.10 + 2.0j
    p3 = -1.10 + 5.0j
    poles = np.array([p1, p2, p3])

    fig, ax = plt.subplots()
    fig.set_size_inches(3.2, 2.6)

    # axis limits (room for annotations)
    xmin, xmax = -2.2, 0.6
    ymin, ymax = -0.8, 6.2
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    # shade stable half-plane Re<0
    ax.axvspan(xmin, 0.0, alpha=0.10)

    # Re=0 boundary
    ax.axvline(0.0, linestyle="--", linewidth=1.0)

    # axes
    ax.axhline(0.0, linewidth=0.8)
    ax.set_xlabel(r"$\Re(p)$")
    ax.set_ylabel(r"$\Im(p)$")

    # plot poles
    ax.scatter(poles.real, poles.imag, s=28, zorder=3)

    # short labels (keep inside)
    labels = [r"$p_1$", r"$p_2$", r"$p_3$"]
    for (pr, pi), lab in zip(zip(poles.real, poles.imag), labels):
        ax.text(pr + 0.06, pi + 0.12, lab, va="bottom")

    # directional annotations (short, outside dense region)
    # 1) Re more negative -> faster decay
    ax.annotate(
        r"$\Re(p)$ more negative $\rightarrow$ faster decay",
        xy=(-1.85, 5.75), xytext=(-0.15, 5.75),
        arrowprops=dict(arrowstyle="<-", linewidth=0.9),
        ha="right", va="center"
    )
    # 2) |Im| larger -> higher oscillation
    ax.annotate(
        r"$|\Im(p)|$ larger $\rightarrow$ higher oscillation",
        xy=(-1.95, 0.35), xytext=(-1.95, 5.35),
        arrowprops=dict(arrowstyle="<-", linewidth=0.9),
        ha="left", va="center", rotation=90
    )

    fig.tight_layout()
    fig.savefig(outpath, bbox_inches="tight")
    plt.close(fig)

def _kernel_modes(t: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Build simple causal kernel modes to visualize pole-residue intuition.
    Returns: (mode1, mode2, mode3, k_sum)
    """
    # three illustrative modes (time domain): r_i * exp(Re_i t) * cos(omega_i t)
    # (These correspond to poles p_i = Re_i ± j*omega_i)
    r1, a1, w1 = 0.80, -2.0, 0.0
    r2, a2, w2 = 0.60, -1.1, 2*np.pi*2.0
    r3, a3, w3 = 0.35, -1.1, 2*np.pi*5.0

    mode1 = r1 * np.exp(a1 * t)  # purely decaying
    mode2 = r2 * np.exp(a2 * t) * np.cos(w2 * t)
    mode3 = r3 * np.exp(a3 * t) * np.cos(w3 * t)
    k_sum = mode1 + mode2 + mode3
    return mode1, mode2, mode3, k_sum

def fig2b_kernel(outpath: str) -> None:
    """
    (b) Kernel components and their sum, drawn as stacked/offset traces.
    No in-axis formula blocks; minimal, clean, paper-friendly.
    """
    t = np.linspace(0.0, 2.0, 900)
    m1, m2, m3, k = _kernel_modes(t)

    # normalize for consistent display height (visualization-only)
    scale = np.max(np.abs(k)) + 1e-9
    m1n, m2n, m3n, kn = m1/scale, m2/scale, m3/scale, k/scale

    fig, ax = plt.subplots()
    fig.set_size_inches(3.2, 2.6)

    offsets = np.array([0.0, 1.6, 3.2, 4.8])
    ax.plot(t, m1n + offsets[0], linewidth=1.1)
    ax.plot(t, m2n + offsets[1], linewidth=1.1)
    ax.plot(t, m3n + offsets[2], linewidth=1.1)
    ax.plot(t, kn  + offsets[3], linewidth=1.2)

    ax.set_xlabel(r"time $t$")
    ax.set_yticks(offsets)
    ax.set_yticklabels([r"mode 1", r"mode 2", r"mode 3", r"$k_\phi(t)$"])
    ax.set_xlim(0.0, 2.0)

    # light guide lines to help stacking readability
    for y0 in offsets:
        ax.axhline(y0, linewidth=0.6, alpha=0.25)

    fig.tight_layout()
    fig.savefig(outpath, bbox_inches="tight")
    plt.close(fig)

def fig2c_signal(outpath: str, seed: int = 7) -> None:
    """
    (c) Input vs enhanced signal, with display normalization and an inset zoom.
    The enhancement is produced by convolving x with k_phi(t) (causal kernel).
    """
    rng = np.random.default_rng(seed)

    # synthetic input signal
    fs = 2000.0
    T = 2.0
    n = int(T * fs)
    t = np.arange(n) / fs

    # baseline vibration + impulsive component + noise
    x = (0.6*np.sin(2*np.pi*30*t) +
         0.25*np.sin(2*np.pi*90*t + 0.5) +
         0.10*rng.standard_normal(n))

    # add a localized burst (to make enhancement visible)
    burst_center = int(1.35 * fs)
    burst = np.zeros_like(x)
    burst[burst_center:burst_center+180] = 1.2*np.sin(2*np.pi*220*t[:180]) * np.hanning(180)
    x = x + burst

    # build causal kernel k_phi(t) from modes
    tk = np.linspace(0.0, 0.12, int(0.12*fs), endpoint=False)
    _, _, _, k = _kernel_modes(tk)
    k = k / (np.sum(np.abs(k)) + 1e-9)  # L1 normalize for stable scale

    # convolve (same length) -> "enhanced" signal
    y = np.convolve(x, k, mode="same")

    # display normalization (visualization only)
    x_disp = (x - x.mean()) / (x.std() + 1e-9)
    y_disp = (y - y.mean()) / (y.std() + 1e-9)

    fig, ax = plt.subplots()
    fig.set_size_inches(6.6, 2.6)

    ax.plot(t, x_disp, linewidth=1.0, label="input")
    ax.plot(t, y_disp, linewidth=1.0, label="enhanced")
    ax.set_xlabel("time (s)")
    ax.set_ylabel("amplitude (norm.)")
    ax.set_xlim(0.0, T)

    # highlight a region of interest
    t1, t2 = 1.30, 1.44
    ax.axvspan(t1, t2, alpha=0.10)

    # legend (compact)
    ax.legend(loc="upper right", frameon=False, ncol=2, handlelength=1.6, columnspacing=1.0)

    # inset zoom
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
    axins = inset_axes(ax, width="35%", height="55%", loc="lower left",
                       bbox_to_anchor=(0.05, 0.08, 1, 1), bbox_transform=ax.transAxes, borderpad=0)
    axins.plot(t, x_disp, linewidth=0.9)
    axins.plot(t, y_disp, linewidth=0.9)
    axins.set_xlim(t1, t2)
    # auto y-limits with margin
    seg = (t >= t1) & (t <= t2)
    y_min = min(x_disp[seg].min(), y_disp[seg].min())
    y_max = max(x_disp[seg].max(), y_disp[seg].max())
    pad = 0.15*(y_max - y_min + 1e-9)
    axins.set_ylim(y_min - pad, y_max + pad)
    axins.set_xticks([])
    axins.set_yticks([])
    mark_inset(ax, axins, loc1=2, loc2=4, fc="none", ec="0.35", linewidth=0.8)

    fig.tight_layout()
    fig.savefig(outpath, bbox_inches="tight")
    plt.close(fig)

def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--outdir", type=str, default=".", help="output directory")
    parser.add_argument("--seed", type=int, default=7, help="random seed for the signal example")
    args = parser.parse_args()

    _ensure_dir(args.outdir)
    fig2a_poles(os.path.join(args.outdir, "fig2a_poles.pdf"))
    fig2b_kernel(os.path.join(args.outdir, "fig2b_kernel.pdf"))
    fig2c_signal(os.path.join(args.outdir, "fig2c_signal.pdf"), seed=args.seed)

if __name__ == "__main__":
    main()


usage: ipykernel_launcher.py [-h] [--outdir OUTDIR] [--seed SEED]
ipykernel_launcher.py: error: unrecognized arguments: --f=c:\Users\Administrator\AppData\Roaming\jupyter\runtime\kernel-v3f0b42f9f1c0cddff9356fdf343717108383278ac.json


SystemExit: 2

d:\anaconda3\envs\phm_bench\lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
